In [1]:
import torch
import json
import torch.nn as nn
import numpy as np
import torch.optim as optim
from ModelUtils import *
from PackingUtils import *
from ModelCrossAttention import *
from tqdm import tqdm
from multiprocessing import Pool

In [2]:
def read_dataset(filename, height_limit=100):
	with open(filename, 'r') as f:
		data_dict = json.load(f)
	height_map_list = []
	item_size_list = []
	feasibility_list = []
	labels = []
	data_length = len(data_dict)
	for t in tqdm(range(data_length), desc=f'Reading data {filename}'):
		data = data_dict[t]
		state_encoding = generate_state_encoding(np.array(data['height_map']), np.array(data['new_item']), height_limit)
		shape_ = state_encoding.shape
		height_map = state_encoding[:, :, 0].reshape(1, shape_[0], shape_[1])
		item_size = np.array(data['new_item'])
		feasibility = state_encoding[:, :, -2:].transpose(2, 0, 1)
		label = generate_label(shape_[:2], data["position"], data["rotate"])
		height_map_list.append(height_map)
		item_size_list.append(item_size)
		feasibility_list.append(feasibility)
		labels.append(label)
	return height_map_list, item_size_list, feasibility_list, labels

In [3]:
read_dataset("dataset_map/dataset_32.json", 100)

Reading data dataset_map/dataset_32.json:   0%|          | 0/15 [00:00<?, ?it/s]

[99, 0, 0] = 1.0


Reading data dataset_map/dataset_32.json:  13%|█▎        | 2/15 [00:00<00:03,  4.32it/s]

[99, 0, 0] = 1.0


Reading data dataset_map/dataset_32.json:  20%|██        | 3/15 [00:11<00:56,  4.70s/it]

...

Reading data dataset_map/dataset_32.json:  93%|█████████▎| 14/15 [01:05<00:07,  7.04s/it]

[99, 78, 1] = 1.0


Reading data dataset_map/dataset_32.json: 100%|██████████| 15/15 [01:14<00:00,  4.96s/it]

[50, 78, 0] = 1.0


([array([[[100., 100., 100., ..., 100., 100., 100.],
          [100., 100., 100., ..., 100., 100., 100.],
          [100., 100., 100., ..., 100., 100., 100.],
          ...,
          [  0.,   0.,   0., ...,   0.,   0.,   0.],
          [  0.,   0.,   0., ...,   0.,   0.,   0.],
          [  0.,   0.,   0., ...,   0.,   0.,   0.]]]),
  array([[[ 0.,  0.,  0., ...,  0.,  0.,  0.],
          [ 0.,  0.,  0., ...,  0.,  0.,  0.],
          [ 0.,  0.,  0., ...,  0.,  0.,  0.],
          ...,
          [31., 31., 31., ..., 31., 31., 31.],
          [31., 31., 31., ..., 31., 31., 31.],
          [31., 31., 31., ..., 31., 31., 31.]]]),
  array([[[ 0.,  0.,  0., ...,  0.,  0.,  0.],
          [ 0.,  0.,  0., ...,  0.,  0.,  0.],
          [ 0.,  0.,  0., ...,  0.,  0.,  0.],
          ...,
          [31., 31., 31., ...,  0.,  0.,  0.],
          [31., 31., 31., ...,  0.,  0.,  0.],
          [31., 31., 31., ...,  0.,  0.,  0.]]]),
  array([[[ 0.,  0.,  0., ...,  0.,  0.,  0.],
          [ 0.,  

In [34]:
p = Pool(35)
dataset_nums = 50
data_list = p.map(read_dataset, [f"dataset_map/dataset_{i}.json" for i in range(50)])

Reading data dataset_map/dataset_37.json: 100%|██████████| 49/49 [05:44<00:00,  7.02s/it]


In [35]:
# height_map_list = [data_list[i][0] for i in range(dataset_nums)]
# item_size_list = [data_list[i][1] for i in range(dataset_nums)]
# feasibility_list = [data_list[i][2] for i in range(dataset_nums)]
# labels = [data_list[i][3] for i in range(dataset_nums)]
height_map_list = []
item_size_list = []
feasibility_list = []
labels = []
for i in range(dataset_nums):
	height_map_list.extend(data_list[i][0])
	item_size_list.extend(data_list[i][1])
	feasibility_list.extend(data_list[i][2])
	labels.extend(data_list[i][3])

In [36]:
height_map_list = np.array(height_map_list)
item_size_list = np.array(item_size_list)
feasibility_list = np.array(feasibility_list)
labels = np.array(labels)
print(height_map_list.shape)
print(item_size_list.shape)
print(feasibility_list.shape)
print(labels.shape)

(1325, 1, 100, 100)
(1325, 3)
(1325, 2, 100, 100)
(1325, 100, 100, 2)


In [37]:
np.save("dataset_map/height_map.npy", height_map_list)
np.save("dataset_map/item_size.npy", item_size_list)
np.save("dataset_map/feasibility.npy", feasibility_list)
np.save("dataset_map/labels.npy", labels)